# Vesuvius V11 - Inference & Submission

## Post-Processing Pipeline:
1. Sliding window inference with Gaussian weighting
2. Test-time augmentation (TTA)
3. Surface-aware smoothing (SDF-based)
4. Slice-wise hole filling (catches tunnels)
5. Small component removal
6. Topology validation (component count check)

In [ ]:
!pip install imagecodecs tifffile -q

In [ ]:
# =============================================================================
# IMPORTS & CONFIG
# =============================================================================

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import gc
import warnings
from pathlib import Path
from typing import List, Tuple
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import tifffile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast

from scipy.ndimage import (
    binary_fill_holes, distance_transform_edt,
    gaussian_filter, label, generate_binary_structure
)

warnings.filterwarnings('ignore')

@dataclass
class Config:
    # Paths - UPDATE THESE
    TEST_ROOT: Path = Path("/kaggle/input/vesuvius-challenge-2025/test")
    CHECKPOINT_PATH: Path = Path("/kaggle/input/v11-checkpoint/fold_0/best_model.pth")
    OUTPUT_DIR: Path = Path("/kaggle/working")
    
    # Model (must match training)
    PATCH_SIZE: Tuple[int, int, int] = (192, 192, 192)
    FEATURES: List[int] = field(default_factory=lambda: [32, 64, 128, 256, 320, 320])
    N_BLOCKS: List[int] = field(default_factory=lambda: [1, 2, 3, 4, 6, 6])
    
    # Inference
    OVERLAP: float = 0.5
    TTA_LEVEL: str = "flip"  # "none", "flip", "full"
    USE_BFLOAT16: bool = True
    
    # Post-processing
    THRESHOLD: float = 0.5
    SURFACE_SIGMA: float = 0.5
    MIN_COMPONENT_SIZE: int = 100
    MAX_TUNNEL_DIAMETER: int = 2
    
    DEVICE: str = "cuda"

cfg = Config()
print(f"Test root: {cfg.TEST_ROOT}")
print(f"Checkpoint: {cfg.CHECKPOINT_PATH}")
print(f"TTA: {cfg.TTA_LEVEL}")

In [ ]:
# =============================================================================
# MODEL ARCHITECTURE
# =============================================================================

def get_num_groups(channels, max_groups=32):
    for g in [max_groups, 16, 8, 4, 2, 1]:
        if channels % g == 0:
            return g
    return 1

class HybridConv3d(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid_ch = out_ch // 2
        self.conv_xy = nn.Conv3d(in_ch, mid_ch, kernel_size=(1, 3, 3), padding=(0, 1, 1), bias=False)
        self.conv_z = nn.Conv3d(in_ch, out_ch - mid_ch, kernel_size=(3, 1, 1), padding=(1, 0, 0), bias=False)
        self.norm = nn.GroupNorm(get_num_groups(out_ch), out_ch)
        self.act = nn.LeakyReLU(0.01, inplace=True)
    
    def forward(self, x):
        return self.act(self.norm(torch.cat([self.conv_xy(x), self.conv_z(x)], dim=1)))

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_hybrid=False):
        super().__init__()
        if use_hybrid:
            self.conv = HybridConv3d(in_ch, out_ch)
        else:
            self.conv = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.GroupNorm(get_num_groups(out_ch), out_ch),
                nn.LeakyReLU(0.01, inplace=True),
            )
    
    def forward(self, x):
        return self.conv(x)

class MultiScaleResBlock(nn.Module):
    def __init__(self, channels, scales=2, use_hybrid=False):
        super().__init__()
        self.scales = scales
        self.width = channels // scales
        self.convs = nn.ModuleList([ConvBlock(self.width, self.width, use_hybrid=use_hybrid) for _ in range(scales - 1)])
        self.norm = nn.GroupNorm(get_num_groups(channels), channels)
    
    def forward(self, x):
        splits = torch.chunk(x, self.scales, dim=1)
        outputs = [splits[0]]
        for i, conv in enumerate(self.convs):
            out = conv(splits[i + 1] + outputs[-1]) if i > 0 else conv(splits[i + 1])
            outputs.append(out)
        return x + self.norm(torch.cat(outputs, dim=1))

class AttentionBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.fc1 = nn.Linear(channels, max(channels // reduction, 8))
        self.fc2 = nn.Linear(max(channels // reduction, 8), channels)
        self.conv_spatial = nn.Conv3d(2, 1, kernel_size=7, padding=3, bias=False)
    
    def forward(self, x):
        b, c = x.shape[:2]
        ca = torch.sigmoid(self.fc2(F.relu(self.fc1(self.gap(x).view(b, c))))).view(b, c, 1, 1, 1)
        x_ca = x * ca
        sa = torch.sigmoid(self.conv_spatial(torch.cat([x_ca.mean(1, keepdim=True), x_ca.max(1, keepdim=True)[0]], 1)))
        return x_ca * sa

class SurfaceRefinementBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.edge_conv = nn.Conv3d(in_ch, in_ch, 3, padding=1, bias=False)
        self.refine = nn.Sequential(
            nn.Conv3d(in_ch * 2, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )
    
    def forward(self, x):
        return self.refine(torch.cat([x, torch.abs(self.edge_conv(x))], dim=1))

class TopoPreservingUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=None, n_blocks=None,
                 use_attention=True, use_hybrid_conv=True, use_surface_refinement=True):
        super().__init__()
        features = features or [32, 64, 128, 256, 320, 320]
        n_blocks = n_blocks or [1, 2, 3, 4, 6, 6]
        self.features = features
        
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i-1]
            layers = [ConvBlock(in_channels, feat, use_hybrid=use_hybrid_conv and i > 0)]
            layers.extend([MultiScaleResBlock(feat, scales=2, use_hybrid=use_hybrid_conv) for _ in range(nb)])
            self.encoders.append(nn.Sequential(*layers))
            self.attentions.append(AttentionBlock(feat) if use_attention and i >= 2 else nn.Identity())
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, 2, stride=2))
        
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features)-2, -1, -1):
            self.ups.append(nn.ConvTranspose3d(features[i+1], features[i], 2, stride=2))
            if use_surface_refinement and i == 0:
                self.dec_convs.append(SurfaceRefinementBlock(features[i]*2, features[i]))
            else:
                self.dec_convs.append(nn.Sequential(
                    ConvBlock(features[i]*2, features[i], use_hybrid=use_hybrid_conv),
                    MultiScaleResBlock(features[i], scales=2, use_hybrid=use_hybrid_conv),
                ))
        
        self.final = nn.Conv3d(features[0], out_ch, 1)
    
    def forward(self, x):
        enc_features = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = att(enc(x))
            enc_features.append(x)
            if i < len(self.pools): x = self.pools[i](x)
        
        enc_features = enc_features[::-1]
        x = enc_features[0]
        
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i+1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
        
        return self.final(x)

print("Model architecture loaded")

In [ ]:
# =============================================================================
# NORMALIZATION & INFERENCE
# =============================================================================

def robust_zscore_normalize(img, lower_pct=0.5, upper_pct=99.5):
    """Percentile clipping + Z-score normalization."""
    p_low, p_high = np.percentile(img, [lower_pct, upper_pct])
    img_clipped = np.clip(img, p_low, p_high)
    return ((img_clipped - img_clipped.mean()) / (img_clipped.std() + 1e-8)).astype(np.float32)


def create_gaussian_weight(patch_size, sigma=0.125):
    """3D Gaussian weighting kernel."""
    d, h, w = patch_size
    gz = np.exp(-0.5 * ((np.arange(d) - d/2) / (d * sigma)) ** 2)
    gy = np.exp(-0.5 * ((np.arange(h) - h/2) / (h * sigma)) ** 2)
    gx = np.exp(-0.5 * ((np.arange(w) - w/2) / (w * sigma)) ** 2)
    return (gz[:, None, None] * gy[None, :, None] * gx[None, None, :]).astype(np.float32)


@torch.no_grad()
def sliding_window_inference(model, volume, patch_size, overlap=0.5, device='cuda', use_bf16=True):
    """Sliding window inference with Gaussian weighting."""
    model.eval()
    dtype = torch.bfloat16 if use_bf16 else torch.float32
    
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    sd, sh, sw = int(pd * (1 - overlap)), int(ph * (1 - overlap)), int(pw * (1 - overlap))
    
    # Pad if needed
    pad_d, pad_h, pad_w = max(0, pd - D), max(0, ph - H), max(0, pw - W)
    if pad_d > 0 or pad_h > 0 or pad_w > 0:
        volume = np.pad(volume, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='reflect')
        D, H, W = volume.shape
    
    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    weight_sum = np.zeros((D, H, W), dtype=np.float32)
    gauss = create_gaussian_weight(patch_size)
    
    # Positions
    z_pos = list(range(0, max(1, D - pd + 1), sd))
    if z_pos[-1] + pd < D: z_pos.append(D - pd)
    y_pos = list(range(0, max(1, H - ph + 1), sh))
    if y_pos[-1] + ph < H: y_pos.append(H - ph)
    x_pos = list(range(0, max(1, W - pw + 1), sw))
    if x_pos[-1] + pw < W: x_pos.append(W - pw)
    
    vol_norm = robust_zscore_normalize(volume)
    
    for z in tqdm(z_pos, desc="Inference", leave=False):
        for y in y_pos:
            for x in x_pos:
                patch = vol_norm[z:z+pd, y:y+ph, x:x+pw]
                inp = torch.from_numpy(patch[None, None]).to(device, dtype=dtype)
                
                with autocast(device_type='cuda', dtype=dtype):
                    prob = torch.sigmoid(model(inp)).squeeze().float().cpu().numpy()
                
                pred_sum[z:z+pd, y:y+ph, x:x+pw] += prob * gauss
                weight_sum[z:z+pd, y:y+ph, x:x+pw] += gauss
    
    pred = pred_sum / np.maximum(weight_sum, 1e-8)
    
    # Remove padding
    if pad_d > 0: pred = pred[:-pad_d]
    if pad_h > 0: pred = pred[:, :-pad_h]
    if pad_w > 0: pred = pred[:, :, :-pad_w]
    
    return pred


@torch.no_grad()
def inference_with_tta(model, volume, patch_size, overlap=0.5, device='cuda', use_bf16=True, tta='flip'):
    """Inference with test-time augmentation."""
    preds = []
    
    # Original
    preds.append(sliding_window_inference(model, volume, patch_size, overlap, device, use_bf16))
    
    if tta in ['flip', 'full']:
        for axis in [0, 1, 2]:
            vol_flip = np.flip(volume, axis).copy()
            pred_flip = sliding_window_inference(model, vol_flip, patch_size, overlap, device, use_bf16)
            preds.append(np.flip(pred_flip, axis).copy())
    
    if tta == 'full':
        for k in [1, 2, 3]:
            vol_rot = np.rot90(volume, k=k, axes=(1, 2)).copy()
            pred_rot = sliding_window_inference(model, vol_rot, patch_size, overlap, device, use_bf16)
            preds.append(np.rot90(pred_rot, k=-k, axes=(1, 2)).copy())
    
    return np.mean(preds, axis=0)

print("Inference functions loaded")

In [ ]:
# =============================================================================
# POST-PROCESSING
# =============================================================================

def surface_aware_smoothing(mask, sigma=0.5):
    """Smooth surface via signed distance field."""
    dist_in = distance_transform_edt(mask)
    dist_out = distance_transform_edt(~mask)
    sdf = dist_in - dist_out
    sdf_smooth = gaussian_filter(sdf.astype(np.float32), sigma=sigma)
    return (sdf_smooth > 0).astype(np.uint8)


def slice_wise_hole_fill(mask, axes=[0, 1, 2]):
    """Fill holes in 2D slices (catches tunnels)."""
    filled = mask.copy()
    for axis in axes:
        for i in range(mask.shape[axis]):
            if axis == 0:
                filled[i] = binary_fill_holes(filled[i])
            elif axis == 1:
                filled[:, i, :] = binary_fill_holes(filled[:, i, :])
            else:
                filled[:, :, i] = binary_fill_holes(filled[:, :, i])
    return filled


def fill_small_tunnels(mask, max_diameter=2):
    """Fill small tunnels near surface."""
    from scipy.ndimage import binary_dilation, binary_erosion
    struct = generate_binary_structure(3, 1)
    dilated = binary_dilation(mask, structure=struct, iterations=max_diameter)
    closed = binary_erosion(dilated, structure=struct, iterations=max_diameter)
    dist = distance_transform_edt(~mask)
    near_surface = dist <= max_diameter
    return (mask | (closed & near_surface)).astype(np.uint8)


def remove_small_components(mask, min_size=100):
    """Remove small connected components."""
    struct = generate_binary_structure(3, 3)
    labeled, n = label(mask, structure=struct)
    if n == 0:
        return mask
    sizes = np.bincount(labeled.ravel())
    small = sizes < min_size
    small[0] = False
    mask_clean = mask.copy()
    mask_clean[small[labeled]] = 0
    return mask_clean


def count_components(mask):
    """Count 26-connected components."""
    struct = generate_binary_structure(3, 3)
    _, n = label(mask, structure=struct)
    return n


def topology_safe_op(mask, op_func, **kwargs):
    """Apply operation, revert if components merge."""
    n_before = count_components(mask)
    result = op_func(mask, **kwargs)
    n_after = count_components(result)
    if n_after < n_before:
        print(f"  [REVERT] {op_func.__name__}: {n_before} -> {n_after} components")
        return mask
    return result


def postprocess(pred_prob, threshold=0.5, surface_sigma=0.5, 
                min_size=100, max_tunnel=2, verbose=True):
    """Full post-processing pipeline."""
    if verbose: print("Post-processing:")
    
    # 1. Threshold
    mask = (pred_prob > threshold).astype(np.uint8)
    if verbose: print(f"  1. Threshold: FG={100*mask.mean():.2f}%")
    
    # 2. Remove small components
    n1 = count_components(mask)
    mask = remove_small_components(mask, min_size)
    n2 = count_components(mask)
    if verbose: print(f"  2. Remove small (<{min_size}): {n1} -> {n2} components")
    
    # 3. Surface smoothing
    mask = surface_aware_smoothing(mask, sigma=surface_sigma)
    if verbose: print(f"  3. Surface smooth (σ={surface_sigma}): FG={100*mask.mean():.2f}%")
    
    # 4. Slice-wise hole fill (topology-safe)
    mask = topology_safe_op(mask, slice_wise_hole_fill)
    if verbose: print(f"  4. Hole fill: FG={100*mask.mean():.2f}%")
    
    # 5. Tunnel fill (topology-safe)
    mask = topology_safe_op(mask, fill_small_tunnels, max_diameter=max_tunnel)
    if verbose: print(f"  5. Tunnel fill: FG={100*mask.mean():.2f}%")
    
    # 6. Final cleanup
    mask = remove_small_components(mask, min_size)
    n_final = count_components(mask)
    if verbose: print(f"  6. Final: {n_final} components, FG={100*mask.mean():.2f}%")
    
    return mask

print("Post-processing functions loaded")

In [ ]:
# =============================================================================
# RLE ENCODING
# =============================================================================

def rle_encode_3d(mask):
    """
    Run-length encode a 3D binary mask.
    Returns: string of space-separated start positions and run lengths
    """
    # Flatten in Fortran order (column-major) as per Kaggle convention
    flat = mask.flatten(order='F')
    
    # Find run starts and lengths
    if flat.sum() == 0:
        return ''
    
    # Pad to find transitions
    padded = np.concatenate([[0], flat, [0]])
    transitions = np.diff(padded)
    
    # Start positions (1-indexed for Kaggle)
    starts = np.where(transitions == 1)[0] + 1
    ends = np.where(transitions == -1)[0] + 1
    lengths = ends - starts
    
    # Format as string
    rle_pairs = [f"{s} {l}" for s, l in zip(starts, lengths)]
    return ' '.join(rle_pairs)


def rle_decode_3d(rle_string, shape):
    """Decode RLE string to 3D mask."""
    if not rle_string or rle_string == '':
        return np.zeros(shape, dtype=np.uint8)
    
    values = list(map(int, rle_string.split()))
    starts = values[0::2]
    lengths = values[1::2]
    
    flat = np.zeros(np.prod(shape), dtype=np.uint8)
    for start, length in zip(starts, lengths):
        flat[start-1:start-1+length] = 1
    
    return flat.reshape(shape, order='F')

print("RLE encoding functions loaded")

In [ ]:
# =============================================================================
# LOAD MODEL
# =============================================================================

def load_model(checkpoint_path, device='cuda'):
    """Load trained model."""
    print(f"Loading: {checkpoint_path}")
    
    model = TopoPreservingUNet3D(
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_attention=True,
        use_hybrid_conv=True,
        use_surface_refinement=True,
    ).to(device)
    
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state_dict'].items()}
    model.load_state_dict(state, strict=False)
    model.eval()
    
    print(f"  Epoch: {ckpt.get('epoch', '?')}, Best Dice: {ckpt.get('best_dice', '?')}")
    return model

model = load_model(cfg.CHECKPOINT_PATH, cfg.DEVICE)

# Compile for speed
if hasattr(torch, 'compile'):
    print("Compiling model...")
    model = torch.compile(model, mode='reduce-overhead')

In [ ]:
# =============================================================================
# RUN INFERENCE
# =============================================================================

# Find test volumes
test_files = sorted(cfg.TEST_ROOT.glob("*.tif")) + sorted(cfg.TEST_ROOT.glob("*.tiff"))
print(f"Found {len(test_files)} test volumes")

submissions = []

for vol_path in test_files:
    vol_id = vol_path.stem
    print(f"\n{'='*60}")
    print(f"Processing: {vol_id}")
    print(f"{'='*60}")
    
    # Load volume
    volume = tifffile.imread(str(vol_path)).astype(np.float32)
    print(f"Shape: {volume.shape}")
    
    # Inference with TTA
    print(f"Running inference (TTA={cfg.TTA_LEVEL})...")
    pred_prob = inference_with_tta(
        model, volume, cfg.PATCH_SIZE, cfg.OVERLAP,
        cfg.DEVICE, cfg.USE_BFLOAT16, cfg.TTA_LEVEL
    )
    
    # Post-process
    pred_mask = postprocess(
        pred_prob,
        threshold=cfg.THRESHOLD,
        surface_sigma=cfg.SURFACE_SIGMA,
        min_size=cfg.MIN_COMPONENT_SIZE,
        max_tunnel=cfg.MAX_TUNNEL_DIAMETER,
    )
    
    # RLE encode
    rle = rle_encode_3d(pred_mask)
    submissions.append({'id': vol_id, 'rle': rle})
    
    print(f"RLE length: {len(rle)}")
    
    # Cleanup
    del volume, pred_prob, pred_mask
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("INFERENCE COMPLETE")
print(f"{'='*60}")

In [ ]:
# =============================================================================
# CREATE SUBMISSION
# =============================================================================

submission_df = pd.DataFrame(submissions)
submission_df.to_csv(cfg.OUTPUT_DIR / 'submission.csv', index=False)

print(f"Submission saved: {cfg.OUTPUT_DIR / 'submission.csv'}")
print(f"Samples: {len(submission_df)}")
submission_df.head()